# Generate 600-text dataset from `human_200.csv`

Self-contained Databricks notebook.

Uses only **GPT-4.1** via Azure OpenAI REST calls.

No separate stats file is created. The source human texts are used only to compute temporary prompt metadata in memory.


In [0]:
# Fixed Databricks paths
INPUT_PATH = "/Workspace/Users/aza@tembi.io/eip_etl/human_200.csv"
OUTPUT_DIR = "/Workspace/Users/aza@tembi.io/eip_etl"
MODEL_DEPLOYMENT = "gpt-4.1"

AI_PLAIN_PATH = f"{OUTPUT_DIR}/ai_plain_200.csv"
AI_OBFUSCATED_PATH = f"{OUTPUT_DIR}/ai_obfuscated_200.csv"
DATASET_600_PATH = f"{OUTPUT_DIR}/dataset_600.csv"

print("INPUT_PATH:", INPUT_PATH)
print("AI_PLAIN_PATH:", AI_PLAIN_PATH)
print("AI_OBFUSCATED_PATH:", AI_OBFUSCATED_PATH)
print("DATASET_600_PATH:", DATASET_600_PATH)


INPUT_PATH: /Workspace/Users/aza@tembi.io/eip_etl/human_200.csv
AI_PLAIN_PATH: /Workspace/Users/aza@tembi.io/eip_etl/ai_plain_200.csv
AI_OBFUSCATED_PATH: /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
DATASET_600_PATH: /Workspace/Users/aza@tembi.io/eip_etl/dataset_600.csv


In [0]:
%pip install nltk
dbutils.library.restartPython()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.1/794.1 kB 74.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 17.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import re
import time
import string
import requests
import pandas as pd
from collections import Counter
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import string


## Load `human_200.csv`

Expected columns: `text`, `genre`, and either `source_idx` or `index`.


In [0]:
human = pd.read_csv(INPUT_PATH)

human = human[["source_idx", "genre", "text"]].copy()
human["label"] = 0
human["obfuscated"] = False
human["model"] = "human"
human["condition"] = "human"

print(human.shape)
print(human["genre"].value_counts(dropna=False))
human.head()

human_200 = human.copy()


(200, 7)
fiction    67
news       67
essays     66
Name: genre, dtype: int64


## Temporary prompt metadata

Used only so GPT sees genre, approximate word count, stylometric stats, and topic keywords.


In [0]:
STOPWORDS = set(stopwords.words("english"))

def get_words(text):
    tokens = word_tokenize(str(text))
    return [
        token.lower()
        for token in tokens
        if any(ch.isalpha() for ch in token)
    ]

def get_sentences(text):
    return sent_tokenize(str(text))

def extract_topic_keywords(text, top_n=8):
    words = get_words(text)

    content_words = [
        word
        for word in words
        if word not in STOPWORDS
        and word not in string.punctuation
        and len(word) > 2
    ]

    freq = FreqDist(content_words)
    return [word for word, _ in freq.most_common(top_n)]

def compute_stats(text):
    text = str(text)

    words = get_words(text)
    sentences = get_sentences(text)

    approx_word_count = len(words)

    avg_sentence_length = (
        approx_word_count / len(sentences)
        if sentences else 0
    )

    avg_word_length = (
        sum(len(word) for word in words) / approx_word_count
        if approx_word_count else 0
    )

    type_token_ratio = (
        len(set(words)) / approx_word_count
        if approx_word_count else 0
    )

    punct_density = (
        sum(1 for char in text if char in string.punctuation) / len(text)
        if text else 0
    )

    return {
        "avg_sentence_length": round(avg_sentence_length, 3),
        "avg_word_length": round(avg_word_length, 3),
        "type_token_ratio": round(type_token_ratio, 3),
        "punct_density": round(punct_density, 4),
        "approx_word_count": approx_word_count,
        "topic_keywords": extract_topic_keywords(text, top_n=8),
    }

stats_rows = []

for _, row in human_200.iterrows():
    stats = compute_stats(row["text"])
    stats_rows.append({
        "source_idx": row["source_idx"],
        "genre": row["genre"],
        "approx_word_count": stats["approx_word_count"],
        "avg_sentence_length": stats["avg_sentence_length"],
        "avg_word_length": stats["avg_word_length"],
        "type_token_ratio": stats["type_token_ratio"],
        "punct_density": stats["punct_density"],
        "topic_keywords": stats["topic_keywords"],
    })

gen_inputs = pd.DataFrame(stats_rows)

gen_inputs.head()

,source_idx,genre,approx_word_count,avg_sentence_length,avg_word_length,type_token_ratio,punct_density,topic_keywords
0,1116,fiction,784,14.255,3.855,0.375,0.0382,"[father, land, brother, bastard, son, sir, rob..."
1,1368,fiction,774,12.094,4.322,0.508,0.0370,"[thy, thou, edward, warwick, king, richard, ri..."
2,422,fiction,760,9.048,4.145,0.442,0.0489,"[clown, titus, emperor, sir, thou, come, shall..."
3,413,fiction,761,21.139,4.511,0.457,0.0248,"[marfa, petrovna, avdotya, romanovna, would, f..."
4,451,fiction,782,5.793,4.160,0.468,0.0548,"[antony, enobarbus, eros, scarus, enter, like,..."


## GPT-4.1 through Azure OpenAI REST

In [0]:
AOAI_KEY = dbutils.secrets.get("kv-deep-eip-secret", "SERVICES-AZURE-AIFOUNDRY-KEY")
AOAI_ENDPOINT = dbutils.secrets.get("kv-deep-eip-secret", "SERVICES-AZURE-AIFOUNDRY-URL").rstrip("/")
AOAI_VERSION = "2024-10-21"


def call_gpt41(prompt, max_tokens=1000, temperature=0.8, retries=3):
    url = f"{AOAI_ENDPOINT}/openai/deployments/{MODEL_DEPLOYMENT}/chat/completions?api-version={AOAI_VERSION}"
    headers = {
        "api-key": AOAI_KEY,
        "Content-Type": "application/json",
    }
    payload = {
        "messages": [
            {
                "role": "system",
                "content": "You write original English text matching requested genre and constraints. Output plain text only. No markdown. No titles unless explicitly requested.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    last_error = None
    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=90)
            if r.status_code == 429 and attempt < retries - 1:
                time.sleep(5 * (attempt + 1))
                continue
            r.raise_for_status()
            body = r.json()
            return body["choices"][0]["message"]["content"].strip()
        except Exception as e:
            last_error = e
            if attempt < retries - 1:
                time.sleep(3 * (attempt + 1))
            else:
                raise last_error


## Prompts

The original human text is never sent to the model.


In [0]:
BANNED_WORDS = [
    "delve", "intricate", "underscore", "meticulous", "testament", "realm", "pivotal",
    "robust", "comprehensive", "seamless", "leverage", "utilize", "nuanced", "multifaceted",
    "tapestry", "embark", "furthermore", "moreover", "notably", "significantly",
]


def feature_block(row):
    return (
        f"Genre: {row['genre']}\n"
        f"Approximate word count: {int(row['approx_word_count'])}\n"
        f"Average sentence length: {row['avg_sentence_length']}\n"
        f"Average word length: {row['avg_word_length']}\n"
        f"Type-token ratio: {row['type_token_ratio']}\n"
        f"Punctuation density: {row['punct_density']}\n"
        f"Topic keywords: {row['topic_keywords']}"
    )


def plain_prompt(row):
    return f"""Write one original {row['genre']} text.

Use the following derived metadata from a human text. Do not ask for or refer to any source text. Do not imitate any specific author or publication. Create a new text that naturally fits the genre, approximate length, style statistics, and topics.

{feature_block(row)}

Requirements:
- Write naturally.
- No markdown.
- No bullet points.
- No title.
- Output text only."""


def obfuscated_prompt(row):
    genre = str(row["genre"]).strip().lower()
    banned = ", ".join(BANNED_WORDS)

    if genre == "fiction":
        technique = "fiction_disfluency_humanized"
        instructions = """Make it sound like a human wrote it casually and imperfectly.
- Add disfluencies, including false starts and self-corrections.
- Include filler phrases such as "the thing is" and "and honestly".
- Include at least two rhetorical questions.
- Use heavy contractions.
- Use wildly varied sentence length."""
    elif genre == "news":
        technique = "translated_news_awkward_plain"
        instructions = """Write as if the text was translated from another language.
- Use slightly awkward phrasing.
- Use plain vocabulary.
- Avoid smooth transitions.
- Prefer short sentences.
- Keep it news-like, but not polished."""
    elif genre == "essays":
        technique = "essay_mixed_lengths_first_person"
        instructions = """Make it sound human and uneven.
- Mix very short sentences under 5 words with long sentences.
- Include a personal first-person anecdote.
- Do not use these phrases: "in conclusion", "furthermore", "moreover", "it is worth noting"."""
    else:
        technique = "generic_humanized"
        instructions = """Make it sound human and uneven.
- Use varied sentence length.
- Include small imperfections in rhythm and phrasing.
- Avoid sounding polished or template-like."""

    prompt = f"""Write one original {row['genre']} text.

Use the following derived metadata from a human text. Do not ask for or refer to any source text. Do not imitate any specific author or publication. Create a new text that fits the genre, approximate length, style statistics, and topics.

{feature_block(row)}

Obfuscation / humanization instructions:
{instructions}

Avoid all of these words and phrases: {banned}.

Requirements:
- No markdown.
- No bullet points.
- No title.
- Output text only."""
    return prompt, technique


## Generate `ai_plain_200.csv`


In [0]:
def load_existing(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def save_csv(df, path):
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} rows -> {path}")


plain_existing = load_existing(AI_PLAIN_PATH)
plain_done = set(plain_existing["source_idx"].tolist()) if not plain_existing.empty and "source_idx" in plain_existing.columns else set()
plain_rows = plain_existing.to_dict("records") if not plain_existing.empty else []

for _, row in gen_inputs.iterrows():
    source_idx = row["source_idx"]
    if source_idx in plain_done:
        continue

    text = call_gpt41(plain_prompt(row), max_tokens=1000, temperature=0.8)
    plain_rows.append({
        "source_idx": source_idx,
        "genre": row["genre"],
        "model": MODEL_DEPLOYMENT,
        "text": text,
        "label": 1,
        "obfuscated": False,
    })
    print(source_idx, text)
    if len(plain_rows) % 10 == 0:
        save_csv(pd.DataFrame(plain_rows), AI_PLAIN_PATH)

ai_plain = pd.DataFrame(plain_rows)
save_csv(ai_plain, AI_PLAIN_PATH)
ai_plain.head()


1116 Jon stood at the far end of the pasture, hands deep in the pockets of his old coat, boots pressing muddy prints into the fresh spring grass. The morning was pale and cold, and the dew on the fence posts shivered in the wind. For a long moment, he watched his father, Sir Robert, checking the stone wall, pressing at loose rocks, shifting his weight from heel to toe as he considered what needed fixing. Robert’s back was broad, bent with age and years of stubbornness, but his voice rang out as clear as ever when he called for his younger son.

“You there, Sam! Check the west corner, the sheep keep getting through. And bring that damn hammer. Can’t do everything myself.”

Jon’s brother, Sam, hurried over, tripping a little in the long grass. Sam had their mother’s nose and their father’s anxious frown, but lacked the weight in his shoulders that Jon bore, the heaviness passed along like a secret between firstborn and father. Jon watched him scramble, the tools jangling at his side, and

,source_idx,genre,model,text,label,obfuscated
0,1116,fiction,gpt-4.1,"Jon stood at the far end of the pasture, hands...",1,False
1,1368,fiction,gpt-4.1,"The candle guttered, casting tall shadows on t...",1,False
2,422,fiction,gpt-4.1,"The morning is bright, but the pale, cold ligh...",1,False
3,413,fiction,gpt-4.1,"Marfa Petrovna had always believed, even as a ...",1,False
4,451,fiction,gpt-4.1,The room at the top of the palace was open to ...,1,False


## Generate `ai_obfuscated_200.csv`

Safe to rerun: if the file already exists, completed `source_idx` rows are skipped.


In [0]:
obf_existing = load_existing(AI_OBFUSCATED_PATH)
obf_done = set(obf_existing["source_idx"].tolist()) if not obf_existing.empty and "source_idx" in obf_existing.columns else set()
obf_rows = obf_existing.to_dict("records") if not obf_existing.empty else []

for _, row in gen_inputs.iterrows():
    source_idx = row["source_idx"]
    if source_idx in obf_done:
        continue

    prompt, technique = obfuscated_prompt(row)
    text = call_gpt41(prompt, max_tokens=1000, temperature=0.9)
    obf_rows.append({
        "source_idx": source_idx,
        "genre": row["genre"],
        "model": MODEL_DEPLOYMENT,
        "text": text,
        "label": 1,
        "obfuscated": True,
        "technique": technique,
    })

    if len(obf_rows) % 10 == 0:
        save_csv(pd.DataFrame(obf_rows), AI_OBFUSCATED_PATH)

ai_obfuscated = pd.DataFrame(obf_rows)
save_csv(ai_obfuscated, AI_OBFUSCATED_PATH)
ai_obfuscated.head()


Saved 10 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 20 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 30 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 40 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 50 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 60 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 70 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 80 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 90 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 100 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 110 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 120 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200.csv
Saved 130 rows -> /Workspace/Users/aza@tembi.io/eip_etl/ai_obfuscated_200

,source_idx,genre,model,text,label,obfuscated,technique
0,1116,fiction,gpt-4.1,"Okay, so, honestly, I never really liked talki...",1,True,fiction_disfluency_humanized
1,1368,fiction,gpt-4.1,"So you wanna hear about Warwick, right? I mean...",1,True,fiction_disfluency_humanized
2,422,fiction,gpt-4.1,"So, right, Titus is scrubbing his face in a cr...",1,True,fiction_disfluency_humanized
3,413,fiction,gpt-4.1,"So, I guess, if you want to know how it all st...",1,True,fiction_disfluency_humanized
4,451,fiction,gpt-4.1,"So, like, here's the thing about Antony—it was...",1,True,fiction_disfluency_humanized


## Merge into `dataset_600.csv`

Adds `condition` and `doc_id`.


In [0]:
human_out = human[["source_idx", "genre", "model", "text", "label", "obfuscated", "condition"]].copy()
human_out["technique"] = ""

plain_out = pd.read_csv(AI_PLAIN_PATH)
plain_out["condition"] = "ai_plain"
plain_out["technique"] = ""

obf_out = pd.read_csv(AI_OBFUSCATED_PATH)
obf_out["condition"] = "ai_obfuscated"

common_cols = ["source_idx", "genre", "model", "text", "label", "obfuscated", "condition", "technique"]

dataset_600 = pd.concat([
    human_out[common_cols],
    plain_out[common_cols],
    obf_out[common_cols],
], ignore_index=True)

dataset_600["doc_id"] = [f"doc_{i:04d}" for i in range(len(dataset_600))]
dataset_600 = dataset_600[["doc_id"] + common_cols]

save_csv(dataset_600, DATASET_600_PATH)

print(dataset_600.shape)
print(dataset_600["condition"].value_counts())
print(dataset_600.groupby(["condition", "genre"]).size())
dataset_600.head()

Saved 600 rows -> /Workspace/Users/aza@tembi.io/eip_etl/dataset_600.csv
(600, 9)
human            200
ai_plain         200
ai_obfuscated    200
Name: condition, dtype: int64
condition      genre  
ai_obfuscated  essays     66
               fiction    67
               news       67
ai_plain       essays     66
               fiction    67
               news       67
human          essays     66
               fiction    67
               news       67
dtype: int64


,doc_id,source_idx,genre,model,text,label,obfuscated,condition,technique
0,doc_0000,1116,fiction,human,"BASTARD. I know not why, except to get the lan...",0,False,human,
1,doc_0001,1368,fiction,human,"I am a king, and privileged to speak. CLIFFORD...",0,False,human,
2,doc_0002,422,fiction,human,"TITUS. Why, didst thou not come from heaven? C...",0,False,human,
3,doc_0003,413,fiction,human,After many tears an unwritten contract was dra...,0,False,human,
4,doc_0004,451,fiction,human,"Alarum. Enter Enobarbus. ENOBARBUS. Naught, na...",0,False,human,


In [0]:
assert len(dataset_600) == 600, f"Expected 600 rows, got {len(dataset_600)}"
assert dataset_600["doc_id"].is_unique
assert dataset_600["condition"].value_counts().to_dict() == {
    "human": 200,
    "ai_plain": 200,
    "ai_obfuscated": 200,
}
assert set(dataset_600["label"].unique()).issubset({0, 1})
print("All checks passed.")

All checks passed.
